## X-Ray Abnormality Detection | Evaluation

> **Antonopoulos Ilias** ( *p3352004* ) <br />
> **Ndoja Silva** ( *p3352017* ) <br />
> **MSc in Data Science, AUEB**

## Table of Contents


* [CNN architecture with sensible defaults](#CNN-architecture-with-sensible-defaults,-all-study-types)
* [DenseNet169 architecture, pretrained on ImageNet](#DenseNet169-architecture,-pretrained-on-ImageNet)
* [DenseNet201 architecture, pretrained on ImageNet](#DenseNet201-architecture,-pretrained-on-ImageNet)
* [VGG19 architecture, pretrained on ImageNet](#VGG19-architecture,-pretrained-on-ImageNet)
* [ResNet50 architecture, pretrained on ImageNet](#ResNet50-architecture,-pretrained-on-ImageNet)
* [Ensemble model: generic CNN + wrist CNN](#Ensemble-model:-generic-CNN-+-wrist-CNN)
* [Ensemble model: DenseNet169 + VGG19](#Ensemble-model:-DenseNet169-+-VGG19)
* [Ensemble model: DenseNet169 + ResNet50](#Ensemble-model:-DenseNet169-+-ResNet50)
* [Ensemble model: DenseNet201 + ResNet50](#Ensemble-model:-DenseNet201-+-ResNet50)

In [22]:
import os
import pathlib
import random
import re

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn import metrics
from sklearn.utils import shuffle

from utils import (
    clean_up,
    F1Score,
    inspect_df,
    study_oriented_transformation,
    study_oriented_transformation_on_ensemble,
)



In [23]:
print(tf.__version__)

2.16.1


In [24]:
SEED = 99910123

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [25]:
DATASET_DIR = "../data/MURA-v1.1/"
IMAGE_WIDTH = 224
IMAGE_HEIGHT = 224

In [26]:
STUDY_TYPES = [
    "XR_ELBOW",
    "XR_HAND",
]

In [27]:
!pip install tensorflow-addons


Defaulting to user installation because normal site-packages is not writeable


In [28]:
METRICS = [
    tf.keras.metrics.TruePositives(name="tp"),
    tf.keras.metrics.FalsePositives(name="fp"),
    tf.keras.metrics.TrueNegatives(name="tn"),
    tf.keras.metrics.FalseNegatives(name="fn"),
    tf.keras.metrics.BinaryAccuracy(name="binary_acc"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
    F1Score(name="f1_score"),
    tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
    tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
]


In [29]:
test_image_paths = pd.read_csv(
    os.path.join(DATASET_DIR, "valid_image_paths.csv"),
    names=["image_path"],
    header=None,
    index_col=False,
)

In [30]:
test_image_paths["image_path"] = test_image_paths["image_path"].map(
    lambda x: os.path.join("../data/", x)
)
test_image_paths["study_type"] = test_image_paths["image_path"].map(
    lambda x: x.split("/")[4]
)
test_image_paths["patient"] = test_image_paths["image_path"].map(
    lambda x: x.split("/")[5]
)
test_image_paths["study"] = test_image_paths["image_path"].map(
    lambda x: x.split("/")[6]
)
test_image_paths["study_path"] = test_image_paths["image_path"].map(
    lambda x: re.sub(r"image\d+.png", "", x)
)

In [31]:
test_labeled_studies = pd.read_csv(
    os.path.join(DATASET_DIR, "valid_labeled_studies.csv"),
    names=["study_path", "label"],
    header=None,
    index_col=False,
)

In [32]:
test_labeled_studies["study_path"] = test_labeled_studies["study_path"].map(
    lambda x: os.path.join("../data/", x)
)
test_labeled_studies["label"] = test_labeled_studies["label"].map(str)

In [33]:
ref_testset = pd.merge(
    test_image_paths, test_labeled_studies, how="inner", on="study_path"
)

inspect_df(ref_testset)

şekil: (925, 6)


,image_path,study_type,patient,study,study_path,label
0,../data/MURA-v1.1/valid/XR_HAND/patient11497/s...,XR_HAND,patient11497,study1_positive,../data/MURA-v1.1/valid/XR_HAND/patient11497/s...,1
1,../data/MURA-v1.1/valid/XR_HAND/patient11497/s...,XR_HAND,patient11497,study1_positive,../data/MURA-v1.1/valid/XR_HAND/patient11497/s...,1
2,../data/MURA-v1.1/valid/XR_HAND/patient11497/s...,XR_HAND,patient11497,study1_positive,../data/MURA-v1.1/valid/XR_HAND/patient11497/s...,1
3,../data/MURA-v1.1/valid/XR_HAND/patient11498/s...,XR_HAND,patient11498,study1_positive,../data/MURA-v1.1/valid/XR_HAND/patient11498/s...,1
4,../data/MURA-v1.1/valid/XR_HAND/patient11498/s...,XR_HAND,patient11498,study1_positive,../data/MURA-v1.1/valid/XR_HAND/patient11498/s...,1


### DenseNet169 architecture, pretrained on ImageNet

In [34]:
densenet_testing = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.densenet.preprocess_input,
)

In [35]:
testset = densenet_testing.flow_from_dataframe(
    dataframe=ref_testset,
    x_col="image_path",
    y_col="label",
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    class_mode="binary",
    batch_size=32,
    seed=SEED,
    shuffle=False,  # this is very important in order to properly use raw predictions to evaluate on study-level!
)

Found 925 validated image filenames belonging to 2 classes.


In [36]:
model = tf.keras.models.load_model(
    "../models/densenet169_pt_imagenet-all-study-types-12644545.h5",
    custom_objects={"F1Score": F1Score},
)

In [37]:
evaluation_metrics = model.evaluate(testset, verbose=1)

C:\Users\tosh\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


29/29 ━━━━━━━━━━━━━━━━━━━━ 131s 4s/step - binary_acc: 0.5447 - f1_score: 0.4311 - fn: 184.5000 - fp: 2.8667 - loss: 1.5869 - pr_auc: 0.8749 - precision: 0.9772 - recall: 0.2843 - roc_auc: 0.7416 - tn: 223.7667 - tp: 83.6000


In [38]:
loss = evaluation_metrics.pop(0)

print("=" * 32)
print(f"test loss   : {loss}")
for metric, score in zip(METRICS, evaluation_metrics):

    print(f"{metric.name + ' ' * (12 - len(metric.name))}: {score}")
print("=" * 32)

test loss   : 0.9370431900024414
tp          : 160.0
fp          : 10.0
tn          : 496.0
fn          : 259.0
binary_acc  : 0.7091891765594482
precision   : 0.9411764740943909
recall      : 0.3818615674972534
f1_score    : <KerasVariable shape=(), dtype=float32, path=f1_score/f1>
roc_auc     : 0.8069349527359009
pr_auc      : 0.818993091583252


In [39]:
# get predictions for the test dataset
preds = model.predict(testset, verbose=1)

29/29 ━━━━━━━━━━━━━━━━━━━━ 147s 5s/step


In [42]:
temp_testset = ref_testset.copy()

temp_testset["label"] = temp_testset["label"].map(int)
temp_testset["prediction"] = preds.ravel()

In [43]:
temp_testset = pd.DataFrame(
    [*study_oriented_transformation(temp_testset)],
    columns=["study_type", "study", "label", "prediction"],
)

In [44]:
k = metrics.cohen_kappa_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
roc_auc = metrics.roc_auc_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
f1 = metrics.f1_score(temp_testset["label"].values, temp_testset["prediction"].values)

print("=" * 38)
print(f"study type       : all studies")
print("-----------------")
print(f"Cohen's kappa (κ): {k}")
print(f"ROC AUC          : {roc_auc}")
print(f"F1               : {f1}")
print("=" * 38)

study type       : all studies
-----------------
Cohen's kappa (κ): 0.33337697477524664
ROC AUC          : 0.6499254200031401
F1               : 0.4777777777777778


In [45]:
for study_type in STUDY_TYPES:

    temp = temp_testset[temp_testset["study_type"] == study_type]

    k = metrics.cohen_kappa_score(temp["label"].values, temp["prediction"].values)
    roc_auc = metrics.roc_auc_score(temp["label"].values, temp["prediction"].values)
    f1 = metrics.f1_score(temp["label"].values, temp["prediction"].values)
    print("=" * 38)
    print(f"study type       : {study_type}")
    print("-----------------")
    print(f"Cohen's kappa (κ): {k}")
    print(f"ROC AUC          : {roc_auc}")
    print(f"F1               : {f1}")

study type       : XR_ELBOW
-----------------
Cohen's kappa (κ): 0.456517904392309
ROC AUC          : 0.7131093544137023
F1               : 0.6138613861386139
study type       : XR_HAND
-----------------
Cohen's kappa (κ): 0.19970375533676044
ROC AUC          : 0.585958595859586
F1               : 0.3037974683544304


In [46]:
clean_up(model)
del temp, temp_testset

### DenseNet201 architecture, pretrained on ImageNet

In [47]:
densenet_testing = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.densenet.preprocess_input,
)

In [48]:
testset = densenet_testing.flow_from_dataframe(
    dataframe=ref_testset,
    x_col="image_path",
    y_col="label",
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    class_mode="binary",
    batch_size=32,
    seed=SEED,
    shuffle=False,  # this is very important in order to properly use raw predictions to evaluate on study-level!
)

Found 925 validated image filenames belonging to 2 classes.


In [49]:
model = tf.keras.models.load_model(
    "../models/densenet201_pt_imagenet-all-study-types-18323905.h5",
    custom_objects={"F1Score": F1Score},
)

In [50]:
evaluation_metrics = model.evaluate(testset, verbose=1)

C:\Users\tosh\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


29/29 ━━━━━━━━━━━━━━━━━━━━ 159s 5s/step - binary_acc: 0.6438 - f1_score: 0.6028 - fn: 147.7667 - fp: 5.7667 - loss: 0.9646 - pr_auc: 0.9081 - precision: 0.9641 - recall: 0.4417 - roc_auc: 0.8063 - tn: 220.8667 - tp: 120.3333


In [51]:
loss = evaluation_metrics.pop(0)

print("=" * 32)
print(f"test loss   : {loss}")
for metric, score in zip(METRICS, evaluation_metrics):

    print(f"{metric.name + ' ' * (12 - len(metric.name))}: {score}")
print("=" * 32)

test loss   : 0.6508013010025024
tp          : 201.0
fp          : 16.0
tn          : 490.0
fn          : 218.0
binary_acc  : 0.7470270395278931
precision   : 0.9262672662734985
recall      : 0.4797135889530182
f1_score    : <KerasVariable shape=(), dtype=float32, path=f1_score/f1>
roc_auc     : 0.8724448084831238
pr_auc      : 0.8672345280647278


In [52]:
# get predictions for the test dataset
preds = model.predict(testset, verbose=1)

29/29 ━━━━━━━━━━━━━━━━━━━━ 164s 5s/step


In [53]:
temp_testset = ref_testset.copy()

temp_testset["label"] = temp_testset["label"].map(int)
temp_testset["prediction"] = preds.ravel()

In [54]:
temp_testset = pd.DataFrame(
    [*study_oriented_transformation(temp_testset)],
    columns=["study_type", "study", "label", "prediction"],
)

In [55]:
k = metrics.cohen_kappa_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
roc_auc = metrics.roc_auc_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
f1 = metrics.f1_score(temp_testset["label"].values, temp_testset["prediction"].values)

print("=" * 38)
print(f"study type       : all studies")
print("-----------------")
print(f"Cohen's kappa (κ): {k}")
print(f"ROC AUC          : {roc_auc}")
print(f"F1               : {f1}")
print("=" * 38)

study type       : all studies
-----------------
Cohen's kappa (κ): 0.4583796256730195
ROC AUC          : 0.7105314806092009
F1               : 0.6020408163265306


In [56]:
for study_type in STUDY_TYPES:

    temp = temp_testset[temp_testset["study_type"] == study_type]

    k = metrics.cohen_kappa_score(temp["label"].values, temp["prediction"].values)
    roc_auc = metrics.roc_auc_score(temp["label"].values, temp["prediction"].values)
    f1 = metrics.f1_score(temp["label"].values, temp["prediction"].values)
    print("=" * 38)
    print(f"study type       : {study_type}")
    print("-----------------")
    print(f"Cohen's kappa (κ): {k}")
    print(f"ROC AUC          : {roc_auc}")
    print(f"F1               : {f1}")

study type       : XR_ELBOW
-----------------
Cohen's kappa (κ): 0.5611111111111111
ROC AUC          : 0.7661396574440053
F1               : 0.7037037037037038
study type       : XR_HAND
-----------------
Cohen's kappa (κ): 0.34854138398914514
ROC AUC          : 0.6541404140414041
F1               : 0.47727272727272724


In [57]:
clean_up(model)
del temp, temp_testset

### ResNet50 architecture, pretrained on ImageNet

In [58]:
resnet_testing = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
)

In [59]:
testset = resnet_testing.flow_from_dataframe(
    dataframe=ref_testset,
    x_col="image_path",
    y_col="label",
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    class_mode="binary",
    batch_size=32,
    seed=SEED,
    shuffle=False,  # this is very important in order to properly use raw predictions to evaluate on study-level!
)

Found 925 validated image filenames belonging to 2 classes.


In [60]:
model = tf.keras.models.load_model(
    "../models/resnet50_pt_imagenet-all-study-types-23589761.h5",
    custom_objects={"F1Score": F1Score},
)

In [61]:
evaluation_metrics = model.evaluate(testset, verbose=1)

C:\Users\tosh\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


29/29 ━━━━━━━━━━━━━━━━━━━━ 107s 4s/step - binary_acc: 0.6103 - f1_score: 0.5634 - fn: 149.5000 - fp: 8.7000 - loss: 1.0500 - pr_auc: 0.8843 - precision: 0.9395 - recall: 0.4090 - roc_auc: 0.7767 - tn: 217.9333 - tp: 118.6000


In [62]:
loss = evaluation_metrics.pop(0)

print("=" * 32)
print(f"test loss   : {loss}")
for metric, score in zip(METRICS, evaluation_metrics):

    print(f"{metric.name + ' ' * (12 - len(metric.name))}: {score}")
print("=" * 32)

test loss   : 0.6831487417221069
tp          : 216.0
fp          : 23.0
tn          : 483.0
fn          : 203.0
binary_acc  : 0.7556756734848022
precision   : 0.9037656784057617
recall      : 0.5155131220817566
f1_score    : <KerasVariable shape=(), dtype=float32, path=f1_score/f1>
roc_auc     : 0.8387205600738525
pr_auc      : 0.838516116142273


In [63]:
# get predictions for the test dataset
preds = model.predict(testset, verbose=1)

29/29 ━━━━━━━━━━━━━━━━━━━━ 109s 4s/step


In [64]:
temp_testset = ref_testset.copy()

temp_testset["label"] = temp_testset["label"].map(int)
temp_testset["prediction"] = preds.ravel()

In [65]:
temp_testset = pd.DataFrame(
    [*study_oriented_transformation(temp_testset)],
    columns=["study_type", "study", "label", "prediction"],
)

In [66]:
k = metrics.cohen_kappa_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
roc_auc = metrics.roc_auc_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
f1 = metrics.f1_score(temp_testset["label"].values, temp_testset["prediction"].values)

print("=" * 38)
print(f"study type       : all studies")
print("-----------------")
print(f"Cohen's kappa (κ): {k}")
print(f"ROC AUC          : {roc_auc}")
print(f"F1               : {f1}")
print("=" * 38)

study type       : all studies
-----------------
Cohen's kappa (κ): 0.4861550294846595
ROC AUC          : 0.7232885853352174
F1               : 0.6224489795918366


In [67]:
for study_type in STUDY_TYPES:

    temp = temp_testset[temp_testset["study_type"] == study_type]

    k = metrics.cohen_kappa_score(temp["label"].values, temp["prediction"].values)
    roc_auc = metrics.roc_auc_score(temp["label"].values, temp["prediction"].values)
    f1 = metrics.f1_score(temp["label"].values, temp["prediction"].values)
    print("=" * 38)
    print(f"study type       : {study_type}")
    print("-----------------")
    print(f"Cohen's kappa (κ): {k}")
    print(f"ROC AUC          : {roc_auc}")
    print(f"F1               : {f1}")

study type       : XR_ELBOW
-----------------
Cohen's kappa (κ): 0.6594240386273496
ROC AUC          : 0.8148880105401844
F1               : 0.7747747747747747
study type       : XR_HAND
-----------------
Cohen's kappa (κ): 0.2998203131684778
ROC AUC          : 0.6314131413141314
F1               : 0.4235294117647058


In [68]:
clean_up(model)
del temp, temp_testset

### Ensemble model: DenseNet169 + ResNet50

In [69]:
densenet_testset = densenet_testing.flow_from_dataframe(
    dataframe=ref_testset,
    x_col="image_path",
    y_col="label",
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    class_mode="binary",
    batch_size=32,
    seed=SEED,
    shuffle=False,  # this is very important in order to properly use raw predictions to evaluate on study-level!
)

Found 925 validated image filenames belonging to 2 classes.


In [70]:
resnet_testset = resnet_testing.flow_from_dataframe(
    dataframe=ref_testset,
    x_col="image_path",
    y_col="label",
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    class_mode="binary",
    batch_size=32,
    seed=SEED,
    shuffle=False,  # this is very important in order to properly use raw predictions to evaluate on study-level!
)

Found 925 validated image filenames belonging to 2 classes.


In [71]:
densenet_model = tf.keras.models.load_model(
    "../models/densenet169_pt_imagenet-all-study-types-12644545.h5",
    custom_objects={"F1Score": F1Score},
)

In [72]:
resnet_model = tf.keras.models.load_model(
    "../models/resnet50_pt_imagenet-all-study-types-23589761.h5",
    custom_objects={"F1Score": F1Score},
)

In [73]:
densenet_preds = densenet_model.predict(densenet_testset, verbose=1)

C:\Users\tosh\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


29/29 ━━━━━━━━━━━━━━━━━━━━ 132s 4s/step


In [74]:
resnet_preds = resnet_model.predict(resnet_testset, verbose=1)

29/29 ━━━━━━━━━━━━━━━━━━━━ 114s 4s/step


In [75]:
temp_testset = ref_testset.copy()

temp_testset["label"] = temp_testset["label"].map(int)
temp_testset["predictionA"] = densenet_preds.ravel()
temp_testset["predictionB"] = resnet_preds.ravel()

In [76]:
temp_testset = pd.DataFrame(
    [*study_oriented_transformation_on_ensemble(temp_testset)],
    columns=["study_type", "study", "label", "prediction"],
)

In [77]:
k = metrics.cohen_kappa_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
roc_auc = metrics.roc_auc_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
f1 = metrics.f1_score(temp_testset["label"].values, temp_testset["prediction"].values)

print("=" * 38)
print(f"study type       : all studies")
print("-----------------")
print(f"Cohen's kappa (κ): {k}")
print(f"ROC AUC          : {roc_auc}")
print(f"F1               : {f1}")
print("=" * 38)

study type       : all studies
-----------------
Cohen's kappa (κ): 0.4105708610415407
ROC AUC          : 0.686607002669179
F1               : 0.5531914893617021


In [78]:
for study_type in STUDY_TYPES:

    temp = temp_testset[temp_testset["study_type"] == study_type]

    k = metrics.cohen_kappa_score(temp["label"].values, temp["prediction"].values)
    roc_auc = metrics.roc_auc_score(temp["label"].values, temp["prediction"].values)
    f1 = metrics.f1_score(temp["label"].values, temp["prediction"].values)
    print("=" * 38)
    print(f"study type       : {study_type}")
    print("-----------------")
    print(f"Cohen's kappa (κ): {k}")
    print(f"ROC AUC          : {roc_auc}")
    print(f"F1               : {f1}")

study type       : XR_ELBOW
-----------------
Cohen's kappa (κ): 0.6031526069634505
ROC AUC          : 0.7867259552042161
F1               : 0.7339449541284403
study type       : XR_HAND
-----------------
Cohen's kappa (κ): 0.19970375533676044
ROC AUC          : 0.585958595859586
F1               : 0.3037974683544304


In [79]:
clean_up(densenet_model)
clean_up(resnet_model)
del temp, temp_testset

### Ensemble model: DenseNet201 + ResNet50

In [80]:
densenet_testset = densenet_testing.flow_from_dataframe(
    dataframe=ref_testset,
    x_col="image_path",
    y_col="label",
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    class_mode="binary",
    batch_size=32,
    seed=SEED,
    shuffle=False,  # this is very important in order to properly use raw predictions to evaluate on study-level!
)

Found 925 validated image filenames belonging to 2 classes.


In [81]:
resnet_testset = resnet_testing.flow_from_dataframe(
    dataframe=ref_testset,
    x_col="image_path",
    y_col="label",
    target_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    class_mode="binary",
    batch_size=32,
    seed=SEED,
    shuffle=False,  # this is very important in order to properly use raw predictions to evaluate on study-level!
)

Found 925 validated image filenames belonging to 2 classes.


In [82]:
densenet_model = tf.keras.models.load_model(
    "../models/densenet201_pt_imagenet-all-study-types-18323905.h5",
    custom_objects={"F1Score": F1Score},
)

In [83]:
resnet_model = tf.keras.models.load_model(
    "../models/resnet50_pt_imagenet-all-study-types-23589761.h5",
    custom_objects={"F1Score": F1Score},
)

In [84]:
densenet_preds = densenet_model.predict(densenet_testset, verbose=1)

C:\Users\tosh\AppData\Roaming\Python\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


29/29 ━━━━━━━━━━━━━━━━━━━━ 178s 6s/step


In [85]:
resnet_preds = resnet_model.predict(resnet_testset, verbose=1)

29/29 ━━━━━━━━━━━━━━━━━━━━ 122s 4s/step


In [86]:
temp_testset = ref_testset.copy()

temp_testset["label"] = temp_testset["label"].map(int)
temp_testset["predictionA"] = densenet_preds.ravel()
temp_testset["predictionB"] = resnet_preds.ravel()

In [87]:
temp_testset = pd.DataFrame(
    [*study_oriented_transformation_on_ensemble(temp_testset)],
    columns=["study_type", "study", "label", "prediction"],
)

In [88]:
k = metrics.cohen_kappa_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
roc_auc = metrics.roc_auc_score(
    temp_testset["label"].values, temp_testset["prediction"].values
)
f1 = metrics.f1_score(temp_testset["label"].values, temp_testset["prediction"].values)

print("=" * 38)
print(f"study type       : all studies")
print("-----------------")
print(f"Cohen's kappa (κ): {k}")
print(f"ROC AUC          : {roc_auc}")
print(f"F1               : {f1}")
print("=" * 38)

study type       : all studies
-----------------
Cohen's kappa (κ): 0.4583796256730195
ROC AUC          : 0.7105314806092009
F1               : 0.6020408163265306


In [89]:
for study_type in STUDY_TYPES:

    temp = temp_testset[temp_testset["study_type"] == study_type]

    k = metrics.cohen_kappa_score(temp["label"].values, temp["prediction"].values)
    roc_auc = metrics.roc_auc_score(temp["label"].values, temp["prediction"].values)
    f1 = metrics.f1_score(temp["label"].values, temp["prediction"].values)
    print("=" * 38)
    print(f"study type       : {study_type}")
    print("-----------------")
    print(f"Cohen's kappa (κ): {k}")
    print(f"ROC AUC          : {roc_auc}")
    print(f"F1               : {f1}")

study type       : XR_ELBOW
-----------------
Cohen's kappa (κ): 0.5903905979951607
ROC AUC          : 0.7812911725955204
F1               : 0.7272727272727273
study type       : XR_HAND
-----------------
Cohen's kappa (κ): 0.31615765227776826
ROC AUC          : 0.638988898889889
F1               : 0.44186046511627913


In [90]:
clean_up(densenet_model)
clean_up(resnet_model)
del temp, temp_testset